In [25]:
# ==========================================================
# Proposed PIPELINE (STRICT WORKFLOW)
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
SPLIT_RATIO = 0.7

HIDDEN = 32
LR_OFFLINE = 0.001
LR_ONLINE = 0.0005

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# DATA
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(np.int64)
    return X, y

def make_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def split_train(X, y):
    split = int(len(X) * SPLIT_RATIO)
    return (X[:split], y[:split]), (X[split:], y[split:])

# ==========================================================
# LOAD CLIENTS
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_offline_loaders = []
client_stream_loaders = []
client_test_loaders = []
client_train_labels = []
client_stream_labels = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))

    (X_off, y_off), (X_stream, y_stream) = split_train(X_train, y_train)

    client_offline_loaders.append(make_loader(X_off, y_off, True))
    client_stream_loaders.append(make_loader(X_stream, y_stream, False))
    client_test_loaders.append(make_loader(X_test, y_test, False))

    client_train_labels.append(y_off)
    client_stream_labels.append(y_stream)
    client_test_labels.append(y_test)

X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global)

input_dim = X_global.shape[1]
num_clients = len(client_offline_loaders)

# ==========================================================
# MODEL
# ==========================================================
class TinyMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# ==========================================================
# EVAL
# ==========================================================
def eval_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for Xb,_ in loader:
            Xb = Xb.to(DEVICE)
            p = torch.sigmoid(model(Xb))
            preds.extend((p >= 0.5).cpu().numpy())
    return np.array(preds)

def acc(p, y):
    return np.mean(p == y) * 100

# ==========================================================
# FEDERATED PRETRAINING (A3)
# ==========================================================
def federated_training(loaders):
    global_model = TinyMLP(input_dim).to(DEVICE)
    global_params = global_model.state_dict()

    for _ in range(FL_ROUNDS):
        agg = {k: torch.zeros_like(v) for k, v in global_params.items()}
        total = 0

        for loader in loaders:
            local = TinyMLP(input_dim).to(DEVICE)
            local.load_state_dict(global_params)

            opt = optim.Adam(local.parameters(), lr=LR_OFFLINE)
            crit = nn.BCEWithLogitsLoss()

            count = 0
            for Xb, yb in loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE).float()
                opt.zero_grad()
                loss = crit(local(Xb), yb)
                loss.backward()
                opt.step()
                count += len(Xb)

            for k in agg:
                agg[k] += local.state_dict()[k] * count
            total += count

        for k in global_params:
            global_params[k] = agg[k] / total

        global_model.load_state_dict(global_params)

    return global_model

# ==========================================================
# ONLINE UPDATE (B2 SIMPLE)
# ==========================================================
def run_online(model, loader):
    model.train()
    opt = optim.Adam(model.parameters(), lr=LR_ONLINE)
    crit = nn.BCEWithLogitsLoss()

    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE).float()

        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        opt.step()

    return model

# ==========================================================
# RUN PIPELINE
# ==========================================================
print("\n=== STEP 1: FEDERATED PRETRAINING ===")
global_model = federated_training(client_offline_loaders)

client_models = []
pretrain_accs = []
online_accs = []
local_test_accs = []

# ==========================================================
# STEP 2–5: CLIENT SIDE
# ==========================================================
for c in range(num_clients):
    print(f"\n===== CLIENT {c} =====")

    model = TinyMLP(input_dim).to(DEVICE)
    model.load_state_dict(global_model.state_dict())

    # ---- PRETRAIN ACC (train only) ----
    p = eval_model(model, client_offline_loaders[c])
    pre_acc = acc(p, client_train_labels[c])
    print(f"Pretrain Train Acc: {pre_acc:.2f}%")

    # ---- ONLINE UPDATE ----
    model = run_online(model, client_stream_loaders[c])

    # ---- POST-ONLINE TRAIN ACC ----
    p2 = eval_model(model, client_stream_loaders[c])
    online_acc = acc(p2, client_stream_labels[c])
    print(f"Online Train Acc: {online_acc:.2f}%")

    # ---- LOCAL TEST ACC ----
    pt = eval_model(model, client_test_loaders[c])
    test_acc = acc(pt, client_test_labels[c])
    print(f"Local Test Acc: {test_acc:.2f}%")

    client_models.append(model)

    pretrain_accs.append(pre_acc)
    online_accs.append(online_acc)
    local_test_accs.append(test_acc)

# ==========================================================
# STEP 6–7: SERVER METRICS
# ==========================================================
print("\n=== SERVER: CLIENT AVERAGES ===")

print(f"Avg Pretrain Train Acc: {np.mean(pretrain_accs):.2f}%")
print(f"Avg Online Train Acc  : {np.mean(online_accs):.2f}%")
print(f"Avg Local Test Acc    : {np.mean(local_test_accs):.2f}%")

# ==========================================================
# STEP 8: AGGREGATION
# ==========================================================
print("\n=== SERVER: AGGREGATION ===")

agg = {k: torch.zeros_like(v) for k,v in global_model.state_dict().items()}

for m in client_models:
    for k in agg:
        agg[k] += m.state_dict()[k]

for k in agg:
    agg[k] /= num_clients

global_model.load_state_dict(agg)

# ==========================================================
# STEP 9: GLOBAL TEST
# ==========================================================
print("\n=== SERVER: GLOBAL TEST ===")

pg = eval_model(global_model, global_loader)
global_acc = acc(pg, y_global)

print(f"Global Test Accuracy: {global_acc:.2f}%")


=== STEP 1: FEDERATED PRETRAINING ===

===== CLIENT 0 =====
Pretrain Train Acc: 74.37%
Online Train Acc: 94.56%
Local Test Acc: 94.09%

===== CLIENT 1 =====
Pretrain Train Acc: 73.28%
Online Train Acc: 99.29%
Local Test Acc: 99.27%

===== CLIENT 2 =====
Pretrain Train Acc: 70.16%
Online Train Acc: 99.98%
Local Test Acc: 99.97%

===== CLIENT 3 =====
Pretrain Train Acc: 53.26%
Online Train Acc: 92.01%
Local Test Acc: 91.31%

===== CLIENT 4 =====
Pretrain Train Acc: 61.77%
Online Train Acc: 84.29%
Local Test Acc: 84.13%

===== CLIENT 5 =====
Pretrain Train Acc: 59.07%
Online Train Acc: 87.73%
Local Test Acc: 87.29%

=== SERVER: CLIENT AVERAGES ===
Avg Pretrain Train Acc: 65.32%
Avg Online Train Acc  : 92.98%
Avg Local Test Acc    : 92.68%

=== SERVER: AGGREGATION ===

=== SERVER: GLOBAL TEST ===
Global Test Accuracy: 82.19%
